# Task 5 — Learning-Rate Sweep across Width (256 / 512 / 1024) → Predict Width 4096

> **Ask.** Sweep the learning rate at widths 256, 512 and 1024, plot loss against
> learning rate, and mark the three minima. State the value you would use at
> width 4096 and how confident you are in it.

### Why the optimum moves

This model uses **standard parametrization (SP)**: `nn.Linear` initialised at
`std=0.02`, LR shared across all tensors. As width grows, the pre-activations into
each matmul grow like $\sqrt{\text{fan\_in}}$ and the Adam update (which is
$O(\eta)$ per element regardless of gradient scale) moves the outputs by more — so
the *same* $\eta$ is effectively "hotter" at larger width and the loss-vs-LR
bowl **shifts left**. Pure SP theory for matmul params predicts
$\eta^\star \propto \text{width}^{-1}$; embeddings / LayerNorms don't scale that
way and pull the exponent toward 0. (**muP** re-scales per-tensor so $\eta^\star$
is width-independent — not what we're doing here, but it's the fix if width keeps
changing.)

### Method (tune every side, per the assignment's warning)

1. **Coarse sweep** — a shared 7-point LR grid at each width, 1 seed, 220 steps →
   locates each basin.
2. **Refinement** — a 5-point grid bracketing each basin, **2 seeds**, 220 steps →
   pins the minimum and gives a seed spread (our error bar). A local quadratic in
   $\log_{10}\eta$ is fit to the *refinement* points only (where the bowl is
   actually quadratic), not to the wide coarse grid.
3. **Extrapolate** $\eta^\star(\text{width})$ as a power law to width 4096.

Cosine schedule, warmup 40, `weight_decay=0.1`, `beta2=0.99`, batch 16, `n_head=8`
throughout. ~35 min on a 4 GB laptop GPU.

In [ ]:
import sys, os, math, time, json, warnings
sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from s11.utils import set_seed, get_device, savefig, plot_style, AIM_REPO
plot_style()
set_seed(1337)
DEVICE = get_device()
torch.set_float32_matmul_precision("high")
print(f"torch {torch.__version__} | device = {DEVICE} "
      f"| {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'cpu'}")
print(f"Aim repo: {AIM_REPO}  (browse it later with:  uv run aim up)")

## 1. Coarse sweep — 3 widths × 7 LRs, locate the basins

In [ ]:
from s11.data import load_char_dataset
from s11.trainer import TrainConfig, train

data = load_char_dataset()
WIDTHS   = [256, 512, 1024]
COARSE   = [3e-4, 6e-4, 1e-3, 2e-3, 4e-3, 8e-3, 1.5e-2]
STEPS    = 220

def run_one(w, lr, seed, tag):
    cfg = TrainConfig(
        n_embd=w, n_head=8, n_layer=6, dropout=0.0,
        total_steps=STEPS, warmup=40, schedule="cosine", min_lr_frac=0.1,
        base_lr=lr, beta2=0.99, weight_decay=0.1, grad_clip=1.0, seed=seed,
        batch_size=16, eval_interval=STEPS-1, eval_iters=80,
        ratio_log_interval=100, label=f"w{w}_lr{lr:.1e}_s{seed}_{tag}", device=DEVICE,
    )
    return train(data, cfg, aim_repo=AIM_REPO, experiment="task5_lr_width",
                 extra_context={"width": w, "stage": tag}, progress=False)

coarse = {}
t0 = time.time()
for w in WIDTHS:
    for lr in COARSE:
        h = run_one(w, lr, 1337, "coarse")
        coarse[(w, lr)] = h["final"]["val_loss"]
        print(f"  w{w:4d} | lr {lr:.1e} | val {h['final']['val_loss']:.4f}")
print(f"\ncoarse wall-clock: {(time.time()-t0)/60:.1f} min")

In [ ]:
# the basin at each width = the 3 grid points around the coarse argmin
basin_center = {}
for w in WIDTHS:
    losses = np.array([coarse[(w, lr)] for lr in COARSE])
    i = int(np.argmin(losses))
    basin_center[w] = COARSE[i]
    print(f"width {w}: coarse best LR = {COARSE[i]:.1e}  (val {losses[i]:.4f})")

## 2. Refinement — 5 LRs per basin × 2 seeds

In [ ]:
REFINE_MULT = [0.5, 0.71, 1.0, 1.41, 2.0]   # geometric bracket around the basin
SEEDS = [1337, 7]

refine = {w: {} for w in WIDTHS}   # width -> {lr: [loss_seed0, loss_seed1]}
t0 = time.time()
for w in WIDTHS:
    lrs = sorted({round(basin_center[w] * m, 6) for m in REFINE_MULT})
    for lr in lrs:
        vals = []
        for s in SEEDS:
            h = run_one(w, lr, s, "refine")
            vals.append(h["final"]["val_loss"])
        refine[w][lr] = vals
        print(f"  w{w:4d} | lr {lr:.2e} | val {np.mean(vals):.4f} ± {np.std(vals):.4f}  {vals}")
print(f"\nrefinement wall-clock: {(time.time()-t0)/60:.1f} min")

## 3. The three minima

In [ ]:
def local_parabola(lrs, losses):
    """Quadratic in log10(lr); minimum clipped to the sampled range."""
    x = np.log10(np.array(lrs)); y = np.array(losses)
    c = np.polyfit(x, y, 2)
    xs = -c[1] / (2 * c[0]) if c[0] > 0 else x[np.argmin(y)]
    xs = float(np.clip(xs, x.min(), x.max()))
    return 10**xs, np.polyval(c, xs), c

minima = {}
rows = []
for w in WIDTHS:
    lrs = sorted(refine[w])
    mean = [float(np.mean(refine[w][lr])) for lr in lrs]
    spread = [float(np.std(refine[w][lr])) for lr in lrs]
    lr_star, loss_star, coef = local_parabola(lrs, mean)
    # seed spread at the grid point nearest the fitted optimum = our error bar
    near = lrs[int(np.argmin([abs(np.log(l) - np.log(lr_star)) for l in lrs]))]
    minima[w] = dict(lr_star=lr_star, loss_star=loss_star, coef=coef,
                     seed_spread=float(np.std(refine[w][near])),
                     grid_best_lr=lrs[int(np.argmin(mean))],
                     grid_best_loss=min(mean))
    rows.append(dict(width=w, fitted_lr_star=lr_star, fitted_loss=loss_star,
                     grid_best_lr=lrs[int(np.argmin(mean))], grid_best_loss=min(mean),
                     seed_spread=minima[w]["seed_spread"]))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 4. Charts — loss vs LR, and the LR-transfer extrapolation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
colors = {256: "C0", 512: "C1", 1024: "C2"}

for w in WIDTHS:
    c = colors[w]
    # coarse (faint)
    lc = sorted(COARSE)
    ax[0].plot(lc, [coarse[(w, lr)] for lr in lc], ":", color=c, alpha=0.4)
    ax[0].scatter(lc, [coarse[(w, lr)] for lr in lc], s=25, color=c, alpha=0.4)
    # refinement (bold, with seed error bars)
    lr_ref = sorted(refine[w])
    m = [np.mean(refine[w][lr]) for lr in lr_ref]
    e = [np.std(refine[w][lr]) for lr in lr_ref]
    ax[0].errorbar(lr_ref, m, yerr=e, fmt="o-", color=c, capsize=3, label=f"width {w}")
    # local parabola
    xs = np.logspace(np.log10(min(lr_ref)), np.log10(max(lr_ref)), 60)
    ax[0].plot(xs, np.polyval(minima[w]["coef"], np.log10(xs)), "--", color=c, alpha=0.6)
    ax[0].scatter([minima[w]["lr_star"]], [minima[w]["loss_star"]], marker="*", s=260,
                  color=c, edgecolor="k", zorder=6)
ax[0].set_xscale("log"); ax[0].set_xlabel("learning rate")
ax[0].set_ylabel("val loss @ step 220"); ax[0].set_ylim(top=min(
    min(np.mean(refine[w][lr]) for lr in refine[w]) for w in WIDTHS) + 0.8)
ax[0].set_title("Loss vs LR  (dotted = coarse, solid = 2-seed refinement, ★ = fitted min)")
ax[0].legend()

ws = np.array(WIDTHS, float)
lrs_star = np.array([minima[w]["lr_star"] for w in WIDTHS])
b, a = np.polyfit(np.log(ws), np.log(lrs_star), 1)
resid = np.log(lrs_star) - (a + b * np.log(ws))
sigma = float(np.sqrt(np.sum(resid**2) / max(1, len(ws) - 2)))
pred = lambda W: math.exp(a + b * math.log(W))
lr_4096 = pred(4096)

ax[1].plot(ws, lrs_star, "o", ms=12, label="fitted optima (2-seed)")
Wg = np.array([256, 512, 1024, 2048, 4096], float)
ax[1].plot(Wg, [pred(W) for W in Wg], "--",
           label=f"power law: η* ∝ width^{b:.2f}")
ax[1].fill_between(Wg, [pred(W)*math.exp(-2*sigma) for W in Wg],
                   [pred(W)*math.exp(2*sigma) for W in Wg], alpha=0.15,
                   label="±2σ fit band")
ax[1].scatter([4096], [lr_4096], color="crimson", s=240, marker="*", edgecolor="k",
              zorder=6, label=f"η*(4096) ≈ {lr_4096:.1e}")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("width (n_embd)"); ax[1].set_ylabel("optimal learning rate")
ax[1].set_title("LR transfer to width 4096"); ax[1].legend()

plt.tight_layout()
savefig(fig, "t5_lr_sweep_width.png")
plt.show()

print(f"power-law exponent b   = {b:+.3f}   (SP theory for matmul params ≈ -1.0)")
print(f"fit residual sigma      = {sigma:.3f}  (in log-LR units)")
print(f"predicted η*(4096)      = {lr_4096:.2e}")
print(f"  ±2σ band             = [{lr_4096*math.exp(-2*sigma):.2e}, {lr_4096*math.exp(2*sigma):.2e}]")
print(f"  endpoint-only (256↔1024) bracket = {math.exp(np.polyval(np.polyfit(np.log(ws[[0,-1]]), np.log(lrs_star[[0,-1]]),1), np.log(4096))):.2e}")

## 5. Sanity — did every run train? (grad-norm + divergence guard)

In [ ]:
guard = []
for w in WIDTHS:
    for lr, vals in refine[w].items():
        guard.append(dict(width=w, lr=lr, mean_val=np.mean(vals), seed_spread=np.std(vals),
                          diverged=bool(any((not np.isfinite(v)) or v > 4.0 for v in vals))))
gd = pd.DataFrame(guard).sort_values(["width", "lr"])
print(gd.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nSeed spread grows with width and with LR — exactly why the 1024 basin "
      "needed 2 seeds to trust.")

## 6. Findings

**The three minima** (2-seed refinement, local quadratic fit — `★` in the left
chart). Numbers below are from the laptop-GPU run; re-running longer will shift
them, so trust the *notebook's printed table*, not these:

| width | fitted $\eta^\star$ | val loss @ 220 | 2-seed spread near $\eta^\star$ |
|---|---|---|---|
| 256  | ~`1.5e-3` | ~2.40 | ±0.002 (flat basin) |
| 512  | ~`1.6e-3` (grid best `2.0e-3`) | ~2.17–2.22 | **±0.03** (noisy basin) |
| 1024 | ~`5e-4` (grid best `4.3e-4`) | ~2.17 | ±0.004 at optimum, ±0.01 just above |

**The trend.** The optimum is essentially **flat from width 256→512** (~`1.5e-3`)
and then **drops sharply at 1024** (~`5e-4`). Fitting all three as
$\eta^\star \propto \text{width}^{\,b}$ gives **b ≈ −0.84** but with a *large* fit
residual (σ ≈ 0.5 in log-LR) precisely because 256 and 512 barely move — SP
theory's $b=-1$ for matmul params is diluted here by the non-scaling
embedding/LayerNorm parameters and by how wide the 256 basin is.

**Value I would use at width 4096:** the power-law fit extrapolates to
**η\*(4096) ≈ `1.8e-4`** (endpoint-only bracket ≈ `1.4e-4`; ±2σ band roughly
`6e-5 … 5e-4`). If instead you extrapolate only the part that is actually
*moving* (512→1024, a ~0.3× drop per doubling) you get ≈ `5e-5` at 4096. So:
**I would start at `2e-4`** and run a tight confirmation sweep
`{1e-4, 2e-4, 4e-4}` at the real training length before committing.

**Confidence: LOW.** Why:

1. **3 width points, 2-parameter fit, σ ≈ 0.5.** The 256 and 512 optima are within
   noise of each other, so the slope is set almost entirely by the 1024 point.
2. **Short runs (220 steps).** The LR that wins at 220 steps sits *above* the one
   that wins at 5–50k steps; a real 4096 run is longer, pushing the true optimum
   *lower* still.
3. **Extrapolating 2 octaves** (1024→4096) from a 2-octave measurement, and the
   width-1024 basin is the sharpest (biggest loss penalty for being wrong).
4. **batch size, warmup, weight decay, β₂ all fixed.** At 4096 several want
   re-tuning — the assignment's "tune both sides" warning applies to the target.

**Honest one-liner:** `2e-4`, good to maybe a factor of 3, as the *center of a
confirmation sweep* — not a final number. Under **muP** the width-1024 optimum
(~`5e-4`) would transfer to 4096 directly and none of this extrapolation would be
needed.